# Traditional RAG Pipeline with ChromaDB (Lectures 25-27)

This notebook implements a complete end-to-end Traditional RAG (Retrieval-Augmented Generation) pipeline using:
- **ChromaDB**: As the local persistent vector store.
- **HuggingFaceEmbeddings**: Dense semantic representations.
- **ChatGroq**: Ultra-fast LLM inference.
- **LangChain Chains**: `create_stuff_documents_chain` and `create_retrieval_chain`.

## 1. Setup & Environment Initialization
Load environment variables and initialize dependencies.

In [1]:
import os
import shutil
import tempfile
from pathlib import Path
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq

try:
    from langchain.chains.combine_documents import create_stuff_documents_chain
    from langchain.chains import create_retrieval_chain
except ImportError:
    from langchain_classic.chains.combine_documents import create_stuff_documents_chain
    from langchain_classic.chains import create_retrieval_chain

load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")
assert groq_api_key, "GROQ_API_KEY must be configured in .env"
print("Environment dependencies loaded.")

Environment dependencies loaded.


## 2. Ingestion & Document Corpus Preparation
Define sample multi-topic technical corpus.

In [2]:
docs_data = [
    Document(
        page_content="Machine learning is a subset of artificial intelligence that learns patterns from data without being explicitly programmed.",
        metadata={"source": "ml_foundations.txt", "topic": "ml"}
    ),
    Document(
        page_content="Deep learning utilizes multi-layered artificial neural networks inspired by biological neural circuits to model complex nonlinear representations.",
        metadata={"source": "deep_learning.txt", "topic": "dl"}
    ),
    Document(
        page_content="Convolutional Neural Networks (CNNs) specialize in processing grid-structured data like images through spatial parameter sharing and pooling layers.",
        metadata={"source": "vision_models.txt", "topic": "cv"}
    ),
    Document(
        page_content="Transformers rely on multi-head self-attention mechanisms to process sequence tokens in parallel, replacing sequential recurrence in modern NLP.",
        metadata={"source": "transformers.txt", "topic": "nlp"}
    ),
]

text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = text_splitter.split_documents(docs_data)
print(f"Generated {len(chunks)} chunks from {len(docs_data)} source documents.")
assert len(chunks) >= len(docs_data), "Chunk count must equal or exceed source documents" 

Generated 4 chunks from 4 source documents.


## 3. Persistent ChromaDB Vector Store Initialization
Store dense embeddings in an isolated persistent directory.

In [3]:
temp_chroma_dir = tempfile.mkdtemp(prefix="chroma_rag_")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=temp_chroma_dir,
    collection_name="rag_mastery_chroma"
)

collection_count = vector_store._collection.count()
print(f"ChromaDB collection initialized with {collection_count} records.")
assert collection_count == len(chunks), f"Expected {len(chunks)} records, got {collection_count}" 

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

ChromaDB collection initialized with 4 records.


## 4. Retriever Configuration & Similarity Search
Configure vector store as retriever with k=2.

In [4]:
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

test_query = "What is deep learning and how does it use neural networks?"
retrieved_docs = retriever.invoke(test_query)

print(f"Retrieved {len(retrieved_docs)} context documents:")
for i, doc in enumerate(retrieved_docs, start=1):
    print(f"  [{i}] {doc.page_content} (topic: {doc.metadata.get('topic')})")

assert len(retrieved_docs) == 2, "Expected 2 retrieved documents"
assert any("Deep learning" in d.page_content for d in retrieved_docs), "Deep learning chunk must be retrieved" 

Retrieved 2 context documents:
  [1] Deep learning utilizes multi-layered artificial neural networks inspired by biological neural circuits to model complex nonlinear representations. (topic: dl)
  [2] Machine learning is a subset of artificial intelligence that learns patterns from data without being explicitly programmed. (topic: ml)


## 5. LLM Initialization & RAG Chain Assembly
Assemble prompt and retrieval chain using ChatGroq.

In [5]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.0,
    max_tokens=256
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a senior AI specialist. Answer questions strictly based on the provided context.\n\nContext:\n{context}"),
    ("human", "{input}")
])

document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
rag_chain = create_retrieval_chain(retriever, document_chain)
print("Traditional RAG chain successfully assembled.")

Traditional RAG chain successfully assembled.


## 6. Pipeline Execution & Verification
Execute queries and verify grounded generation.

In [6]:
query = "What is deep learning and how does it differ from traditional machine learning?"
response = rag_chain.invoke({"input": query})

answer = response["answer"].strip()
print(f"Query: {query}\n")
print(f"Answer:\n{answer}\n")

assert len(answer) > 20, "Answer must not be empty"
print("Context Sources:")
for doc in response["context"]:
    print(f"- {doc.metadata.get('source')}: {doc.page_content[:60]}...")

# Clean up temp directory
shutil.rmtree(temp_chroma_dir, ignore_errors=True)
print("Pipeline verified and resources cleaned up.")

Query: What is deep learning and how does it differ from traditional machine learning?

Answer:
**Deep learning** is a specific type of machine learning that uses *multi‑layered artificial neural networks*—inspired by how biological brains process information—to learn and represent complex, nonlinear patterns in data.

**How it differs from “traditional” machine learning**

| Aspect | Traditional machine learning (as implied by the context) | Deep learning |
|--------|--------------------------------------------------------|---------------|
| **Model architecture** | Typically simpler models (e.g., linear regression, decision trees, support

Context Sources:
- ml_foundations.txt: Machine learning is a subset of artificial intelligence that...
- deep_learning.txt: Deep learning utilizes multi-layered artificial neural netwo...
Pipeline verified and resources cleaned up.
